# Imports

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import itertools

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score
from yellowbrick.classifier import ROCAUC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import confidenceinterval

import shap
from sklearn.dummy import DummyClassifier

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import scipy.stats

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
random_state = 1234
%matplotlib inline

In [ ]:
sns.set_theme(context='notebook', style='white', 
              font = 'arial', font_scale = 1)

# Train: Discovery, Test: Validation

## Data import

In [ ]:
disc = pd.read_csv("output/discovery/Disc_cohort_metadata_20260913.csv")
val = pd.read_csv("output/validation/Val_cohort_metadata_20260913.csv")

In [ ]:
disc['PE_binary_int'] = disc['PE_binary'].map({0.0: 0, 1.0: 1})
val['PE_binary_int'] = val['PE_binary'].map({0.0: 0, 1.0: 1})

In [ ]:
all_mut_sig = pd.read_csv("input/discovery/Discovery_Validation_NMFres_signatures_by_sample_four_sigs_20260302.csv", index_col=0).T
all_mut_sig.columns = ['SBSA', 'SBSB', 'SBSC', 'SBSD']
all_mut_sig.head()

In [ ]:
disc = pd.merge(disc, all_mut_sig, left_on="plac_id", right_index=True)
val = pd.merge(val, all_mut_sig, left_on="SampleID", right_index=True)

In [ ]:
disc.to_csv("output/discovery/discovery_cohort_metadata_SBSABCD_20261101.csv")
val.to_csv("output/validation/validation_cohort_metadata_SBSABCD_20261101.csv")

In [ ]:
sbs_cols = ['SBSA', 'SBSB', 'SBSC', 'SBSD', 'HypoxiaScoreBuffa_new']

for sbs in sbs_cols:
    x = disc['eTMB']
    y = disc[sbs]
    
    print(sbs, scipy.stats.pearsonr(x, y))

In [ ]:
X = disc[["eTMB", "HypoxiaScoreBuffa_new", "SBSA", "SBSB", "SBSC", "SBSD"]]

X_vif = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) 
                   for i in range(X_vif.shape[1])]
vif_data

In [ ]:
test_df = disc
sns.set_style("white") 

colors = ['#508A91', '#B1CCC2', '#6666A3','#B4A8C7']

ax = sns.violinplot(x = 'PE_onset_uniform', y = "SBSD", data = test_df, 
                order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"],
                boxprops={'alpha': 1}, palette = colors, showfliers=False)
sns.stripplot(data=test_df, x="PE_onset_uniform", y="SBSD",
              hue="PE_onset_uniform", hue_order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"],
              order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"], 
              dodge=False, ax=ax,palette='dark:black',alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:4], labels[:4], title='PE_onset_uniform', bbox_to_anchor=(1, 1.02), loc='upper left')
ax.set_xticklabels(["EP-N", "NTT-N", "EP-PE", "NTT-PE"])

plt.legend().remove()
plt.xlabel('Maternal outcome')
plt.ylabel('SBSD')
plt.ylim(-0.03, 0.11)

for spine in ax.spines.values():
    spine.set_color((0, 0, 0, 1.0))
    spine.set_linewidth(1)

ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.savefig("output/figure2/Discovery_SBSD.pdf", format="pdf", bbox_inches="tight")

In [ ]:
# sample sizes 

m1 = disc[disc['variant_cohort'] == 'discovery']
print(m1['variant_cohort'].value_counts())

m2 = val
print(m2['variant_cohort'].value_counts())

## Predictive modeling: Preeclampsia

In [ ]:
feature_cols = ['HypoxiaScoreBuffa_new', 'SBSA', 'SBSB', 'SBSC',  'SBSD']
feature_combos = []

for r in range(1, len(feature_cols)+1):
    for combo in itertools.combinations(feature_cols, r):
        feature_combos.append(list(combo))
# feature_combos

In [ ]:
## model and feature selection
models = {"LogisticRegression":  Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight = "balanced"))]),
          "KNN": Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())]), 
           "SVC": Pipeline([('scaler', StandardScaler()), ('clf', LinearSVC())])
         }

predictor = "PE_binary_int"

cv = KFold(n_splits=5, random_state=random_state, shuffle=True)

best_auc = 0
best_model = None
best_features = None

for feature_cols in feature_combos:
    X = m1[feature_cols] 
    y = m1[predictor] 
    
    for model_name, model in models.items():
        auc_scores = cross_val_score(clone(model), X, y, cv=cv, scoring='accuracy')

        mean_auc = np.mean(auc_scores)
        
        if mean_auc > best_auc:
            best_auc = mean_auc
            best_model = model_name
            best_features = feature_cols

print("Best mean CV AUC:", best_auc)
print("Best model:", best_model)
print("Best feature set:", best_features)

In [ ]:
fig, ax = plt.subplots(figsize = (6,4))

feature_cols = best_features
model =  models[best_model]

X_train = m1[feature_cols].copy()
y_train = m1[predictor]

X_test = m2[feature_cols].copy()
y_test = m2[predictor]

model.fit(X_train, y_train)

visualizer = ROCAUC(model, binary=True,  encoder=None, 
                    is_fitted=True, force_model=False, ax=ax)
visualizer.fit(X_train, y_train)
visualizer.score(X_train, y_train)
visualizer.score(X_test, y_test)     

# Plotting 
visualizer.show("") 
visualizer.ax.grid(False)
visualizer.finalize()   

# Formatting
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.set_xticks(np.linspace(0, 1, 6))
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_xlabel("False positive rate", fontsize=11)
ax.set_ylabel("True positive rate", fontsize=11)
ax.tick_params(axis='both', labelsize=11)
ax.grid(False)
ax.set_title(" ")

for spine in ax.spines.values():
    spine.set_color('black')
ax.lines[0].set_color("#9bbbbc")  
ax.lines[2].set_color("#6e5053")  
ax.get_legend().remove()    


print("Discovery Accuracy", model.score(X_train, y_train))
print("Validation Accuracy", model.score(X_test, y_test))

plt.savefig("output/figure5/Discovery_PM_PE.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
print(confidenceinterval.roc_auc_score(y_train, model.predict_proba(X_train)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_train, model.predict(X_train))

In [ ]:
print(confidenceinterval.roc_auc_score(y_test, model.predict_proba(X_test)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_test, model.predict(X_test))

### Model metrics

In [ ]:
def f(x):
    return model.predict_proba(x)[:, 1]

med = X_train.median().to_frame().T

explainer = shap.Explainer(f, med)

In [ ]:
shap_values = explainer(X_test.iloc[0:1000, :])

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance = pd.DataFrame({"Feature": X_test.columns, "Mean_abs_SHAP": mean_abs_shap})
shap_importance

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print(confidenceinterval.roc_auc_score(y_train, dummy.predict_proba(X_train)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_train, dummy.predict(X_train))

In [ ]:
print(confidenceinterval.roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_test, dummy.predict(X_test))


In [ ]:
def train_evaluate(X_tr, X_te):
        model =  models[best_model]
        model.fit(X_tr, y_train)
        return roc_auc_score(y_test, model.predict_proba(X_te)[:, 1])

def ablation(X_train, X_test, y_train, y_test):
    baseline = train_evaluate(X_train, X_test)
    
    results = {}
    for feature in X_train.columns:
        score = train_evaluate(
            X_train.drop(columns=feature),
            X_test.drop(columns=feature)
        )
        print(feature, score)

        results[feature] = baseline - score

    return baseline, results

In [ ]:
ablation(X_train, X_test, y_train, y_test)

## Predictive modeling: Maternal vascular malperfusion

In [ ]:
feature_cols = ['HypoxiaScoreBuffa_new', 'SBSA', 'SBSB', 'SBSC',  'SBSD']
feature_combos = []

for r in range(1, len(feature_cols)+1):
    for combo in itertools.combinations(feature_cols, r):
        feature_combos.append(list(combo))
# feature_combos

In [ ]:
## model and feature selection
models = {"LogisticRegression":  Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight = "balanced"))]),
          "KNN": Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())]), 
           "SVC": Pipeline([('scaler', StandardScaler()), ('clf', LinearSVC())])
         }

predictor = "Path_MVM_binary"

cv = KFold(n_splits=5, random_state=random_state, shuffle=True)

best_auc = 0
best_model = None
best_features = None

for feature_cols in feature_combos:
    X = m1[feature_cols] 
    y = m1[predictor] 
    
    for model_name, model in models.items():
        auc_scores = cross_val_score(clone(model), X, y, cv=cv, scoring='accuracy')

        mean_auc = np.mean(auc_scores)
        
        if mean_auc > best_auc:
            best_auc = mean_auc
            best_model = model_name
            best_features = feature_cols

print("Best mean CV AUC:", best_auc)
print("Best model:", best_model)
print("Best feature set:", best_features)

In [ ]:
fig, ax = plt.subplots(figsize = (6,4))

feature_cols = best_features
model =  models[best_model]

X_train = m1[feature_cols].copy()
y_train = m1[predictor]

X_test = m2[feature_cols].copy()
y_test = m2[predictor]

model.fit(X_train, y_train)

visualizer = ROCAUC(model, binary=True,  encoder=None, 
                    is_fitted=True, force_model=False, ax=ax)
visualizer.fit(X_train, y_train)
visualizer.score(X_train, y_train)
visualizer.score(X_test, y_test)     

# Plotting 
visualizer.show("") 
visualizer.ax.grid(False)
visualizer.finalize()   

# Formatting
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.set_xticks(np.linspace(0, 1, 6))
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_xlabel("False positive rate", fontsize=11)
ax.set_ylabel("True positive rate", fontsize=11)
ax.tick_params(axis='both', labelsize=11)
ax.grid(False)
ax.set_title(" ")

for spine in ax.spines.values():
    spine.set_color('black')
ax.lines[0].set_color("#9bbbbc")  
ax.lines[2].set_color("#6e5053")  
ax.get_legend().remove()    


print("Discovery Accuracy", model.score(X_train, y_train))
print("Validation Accuracy", model.score(X_test, y_test))

plt.savefig("output/figure5/Discovery_PM_PE.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
print(confidenceinterval.roc_auc_score(y_train, model.predict_proba(X_train)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_train, model.predict(X_train))

In [ ]:
print(confidenceinterval.roc_auc_score(y_test, model.predict_proba(X_test)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_test, model.predict(X_test))

### Model metrics

In [ ]:
def f(x):
    return model.predict_proba(x)[:, 1]

med = X_train.median().to_frame().T

explainer = shap.Explainer(f, med)

In [ ]:
shap_values = explainer(X_test.iloc[0:1000, :])

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance = pd.DataFrame({"Feature": X_test.columns, "Mean_abs_SHAP": mean_abs_shap})
shap_importance

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print(confidenceinterval.roc_auc_score(y_train, dummy.predict_proba(X_train)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_train, dummy.predict(X_train))


In [ ]:
print(confidenceinterval.roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_test, dummy.predict(X_test))

In [ ]:
def train_evaluate(X_tr, X_te):
        model =  models[best_model]
        model.fit(X_tr, y_train)
        probabilities = model.predict_proba(X_te)[:, 1]
        return roc_auc_score(y_test, probabilities)

def ablation(X_train, X_test, y_train, y_test):
    baseline = train_evaluate(X_train, X_test)
    
    results = {}
    for feature in X_train.columns:
        score = train_evaluate(
            X_train.drop(columns=feature),
            X_test.drop(columns=feature)
        )
        print(feature, score)

        results[feature] = baseline - score

    return baseline, results

In [ ]:
ablation(X_train, X_test, y_train, y_test)

# Circulating

## Data import and add metadata

In [ ]:
meta = pd.read_csv("input/circulating/circulating_metadata.csv")
meta.head()

In [ ]:
disc = pd.read_csv("input/circulating/GSE192902_counts_Discovery_postQC.csv")
disc = list(disc.columns[2:len(disc.columns)])
len(disc)

In [ ]:
val1 = pd.read_csv("input/circulating/GSE192902_counts_Validation1_postQC.csv")
val1 = list(val1.columns[2:len(val1.columns)])
len(val1)

In [ ]:
val2 = pd.read_csv("input/circulating/GSE192902_counts_Validation2_postQC.csv")
val2 = list(val2.columns[2:len(val2.columns)])
len(val2)

In [ ]:
conditions = [
    meta["Library Name"].isin(disc),
    meta["Library Name"].isin(val1),
    meta["Library Name"].isin(val2)
]

choices = ["Discovery", "Validation_1", "Validation_2"]

meta["Batch"] = np.select(conditions, choices, default="NA")

In [ ]:
set(meta["Batch"])

In [ ]:
meta["disease_uniform"] = np.where(
    meta["disease"].isna(),
    "Normal",
    "PE"
)
meta.head()
#meta.to_csv("input/circulating/final_metadata_20251228.csv")

In [ ]:
# match names 
meta['Sample'] = 'Sample' + meta['Run']
meta['PE_binary'] = np.where(meta['disease'].isna(), 0, 1)
meta[['Isolate', 'sampling_order']] = meta['Library Name'].str.split('_', expand=True)

In [ ]:
hyp = pd.read_csv("input/circulating/Circulating_Discovery_hypoxia_gene_scoring.csv")
hyp2 = pd.read_csv("input/circulating/Circulating_Validation_1_hypoxia_gene_scoring.csv")
hyp3 = pd.read_csv("input/circulating/Circulating_Validation_2_hypoxia_gene_scoring.csv")
hypoxia = pd.concat([hyp, hyp2, hyp3])

hypoxia = hypoxia.rename(columns={"Unnamed: 0": "Sample"})
hypoxia.head()

In [ ]:
meta = meta.merge(hypoxia[['Sample', 'HypoxiaScoreBuffa']], on='Sample')
meta.shape

In [ ]:
meta.head()

In [ ]:
# read in genome coverage
coverage = pd.read_csv("input/circulating/circulating_coverage_1.txt", sep="\t" )
coverage2 = pd.read_csv("input/circulating/circulating_coverage_2.txt", sep="\t" )
coverage = pd.concat([coverage, coverage2])
coverage.columns = ['File Name', 'Coverage']

coverage['Sample'] = coverage['File Name'].str.split('_', n=1).str[0]
meta = meta.merge(coverage[['Coverage', 'Sample']], on='Sample')

In [ ]:
maf1 = pd.read_csv("input/circulating/Circulating_Discovery_rare_final_maf_maftools.maf", 
                  sep = "\t")
maf2 = pd.read_csv("input/circulating/Circulating_Validation 1_rare_final_maf_maftools.maf", 
                  sep = "\t")
maf3 = pd.read_csv("input/circulating/Circulating_Validation 2_rare_final_maf_maftools.maf", 
                  sep = "\t")
maf = pd.concat([maf1, maf2, maf3])

maf['vaf'] = maf['t_alt_count']/(maf['t_alt_count']+maf['t_ref_count'])
maf

In [ ]:
print(len(maf["Tumor_Sample_Barcode"].value_counts()))
maf['vaf'].describe()

In [ ]:
vaf_df = pd.DataFrame(maf['Tumor_Sample_Barcode'].value_counts())
vaf_df.rename(columns={'count': 'vaf_filtered_mut_count'}, inplace=True)

print(vaf_df.shape)
vaf_df['vaf_filtered_mut_count'].describe()

In [ ]:
merged_df = pd.merge(meta, vaf_df, left_on = 'Sample', right_on = 'Tumor_Sample_Barcode', how="left")
hg38_num_bases=3088286401
merged_df[merged_df['vaf_filtered_mut_count'].isnull()]

In [ ]:
merged_df['vaf_filtered_mut_count'] = merged_df['vaf_filtered_mut_count'].fillna(0.0)
merged_df[merged_df['vaf_filtered_mut_count'].isnull()]

In [ ]:
merged_df['Number_of_bases_per106.megabase.']= merged_df['Coverage']*100*hg38_num_bases/10**6
merged_df['eTMB_VAF_filtered'] = merged_df['vaf_filtered_mut_count']*100/merged_df['Number_of_bases_per106.megabase.']
merged_df['eTMB'] = merged_df['eTMB_VAF_filtered']
merged_df.head()

In [ ]:
sbs = pd.read_csv("input/circulating/Circulating_NMFres_signatures_by_sample_6sigs_20260302.csv",
                 index_col = 0).T
sbs.columns = ['SBSA', 'SBSB', 'SBSC', 'SBSD', 'SBSE', 'SBSF']
sbs

In [ ]:
merged_df = merged_df.merge(sbs, left_on = "Run", right_index= True, how = "left")

In [ ]:
merged_df['SBSA'] = merged_df['SBSA'].fillna(0.0)
merged_df['SBSB'] = merged_df['SBSB'].fillna(0.0)
merged_df['SBSC'] = merged_df['SBSC'].fillna(0.0)
merged_df['SBSD'] = merged_df['SBSD'].fillna(0.0)
merged_df['SBSE'] = merged_df['SBSE'].fillna(0.0)
merged_df['SBSF'] = merged_df['SBSF'].fillna(0.0)

In [ ]:
merged_df.shape

In [ ]:
# merged_df.to_csv("output/circulating/circulating_cohort_metadata_20260913.csv", index= False)

## Predictive modeling: Preeclampsia

In [ ]:
circ_meta = pd.read_csv("output/circulating/circulating_cohort_metadata_20260913.csv", index_col = 0)

# filter for number of coverage
x = circ_meta['Number_of_bases_per106.megabase.']
y = circ_meta['eTMB']
plt.scatter(x, y, color='black', marker='o', alpha=0.5)
plt.show()

In [ ]:
circ_meta = circ_meta[circ_meta['Number_of_bases_per106.megabase.'].between(500, 3000)]

meta =  circ_meta[circ_meta['sampling_order'].isin([1])]
m1 = meta[meta['Batch'] == 'Discovery']
m2 = meta[meta['Batch'] == 'Validation_1']
m3 = meta[meta['Batch'] == 'Validation_2']

print("Discovery PE: ", m1['PE_binary'].value_counts()[0], 
      "Non-PE: ", m1['PE_binary'].value_counts()[1])
print("Validation 1 PE: ", m2['PE_binary'].value_counts()[0], 
      "Non-PE: ", m2['PE_binary'].value_counts()[1])
print("Validation 2 PE: ", m3['PE_binary'].value_counts()[0], 
      "Non-PE: ", m3['PE_binary'].value_counts()[1])

Forgo validation 1, the dataset distribution is too different for this subset.

In [ ]:
import scipy.stats
sbs_cols = ['SBSA', 'SBSB', 'SBSC', 'SBSD', 'SBSE', 'SBSF']

for sbs in sbs_cols:
    x = m1['eTMB']
    y = m1[sbs]
    
    print(sbs, scipy.stats.pearsonr(x, y))

In [ ]:
X = m1[['SBSA', 'SBSB', 'SBSC', 'SBSD', 'SBSE', 'SBSF', 'eTMB']]

X_vif = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) 
                   for i in range(X_vif.shape[1])]
vif_data

In [ ]:
feature_cols = ['HypoxiaScoreBuffa', 'SBSA', 'SBSB', 'SBSC', 'SBSD', 'SBSE', 'SBSF']

feature_combos = []

for r in range(1, len(feature_cols)+1):
    for combo in itertools.combinations(feature_cols, r):
            feature_combos.append(list(combo))

In [ ]:
models = {"LR": Pipeline([('scaler', StandardScaler()), 
                    ('knn', LogisticRegression(class_weight="balanced"))])}

cv = KFold(n_splits=3, random_state=random_state, shuffle=True) 
predictor = "PE_binary"

best_auc = 0
best_model = None
best_features = None


for feature_cols in feature_combos:
    X = m1[feature_cols] 
    y = m1[predictor] 
    
    for model_name, model in models.items():
        auc_scores = cross_val_score(clone(model), X, y, cv=cv, 
                                     scoring='accuracy')
        mean_auc = np.mean(auc_scores)
    
    if mean_auc > best_auc:
        best_auc = mean_auc
        best_model = model_name
        best_features = feature_cols

print("Best mean CV AUC:", best_auc)
print("Best model:", best_model)
print("Best feature set:", best_features)

In [ ]:
feature_cols = best_features
fig, ax = plt.subplots(figsize=(6,4))

X_train = m1[feature_cols]
y_train = m1[predictor]

X_test = m3[feature_cols]
y_test = m3[predictor]

model =  Pipeline([('scaler', StandardScaler()), 
                    ('lr', LogisticRegression(class_weight = "balanced"))])


visualizer = ROCAUC(model, binary=True, encoder=None,
    is_fitted='auto', force_model=False, ax=ax)
visualizer.fit(X_train, y_train)
visualizer.score(X_train, y_train)
visualizer.score(X_test, y_test)
visualizer.finalize()

model.fit(X_train, y_train)
print("Discovery Accuracy (Blue)", model.score(m1[feature_cols], m1[predictor]))
print("Validation Accuracy (Green)", model.score(m3[feature_cols], m3[predictor]))

# Formatting
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.set_xticks(np.linspace(0, 1, 6))
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_xlabel("False positive rate", fontsize=11)
ax.set_ylabel("True positive rate", fontsize=11)
ax.tick_params(axis='both', labelsize=11)
ax.grid(False)
ax.set_title(" ")

for spine in ax.spines.values():
    spine.set_color('black')
ax.lines[0].set_color("#9bbbbc")  
ax.lines[2].set_color("#6e5053")  
ax.get_legend().remove()    

plt.savefig("output/figure5/Circulating_PM_PE.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
print(confidenceinterval.roc_auc_score(y_train, model.predict_proba(X_train)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_train, model.predict(X_train))

In [ ]:
print(confidenceinterval.roc_auc_score(y_test, model.predict_proba(X_test)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_test, model.predict(X_test))

### Model metrics

In [ ]:
def f(x):
    return model.predict_proba(x)[:, 1]

med = X_train.median().to_frame().T

explainer = shap.Explainer(f, med)
shap_values = explainer(X_test.iloc[0:1000, :])

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance = pd.DataFrame({"Feature": X_test.columns, "Mean_abs_SHAP": mean_abs_shap})
shap_importance

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print(confidenceinterval.roc_auc_score(y_train, dummy.predict_proba(X_train)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_train, dummy.predict(X_train))

In [ ]:
print(confidenceinterval.roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1], confidence_interval = 0.9))
confidenceinterval.classification_report_with_ci(y_test, dummy.predict(X_test))

In [ ]:
def train_evaluate(X_tr, X_te):
    model =  models[best_model]
    model.fit(X_tr, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_te)[:, 1])

def ablation(X_train, X_test, y_train, y_test):
    baseline = train_evaluate(X_train, X_test)
    
    results = {}
    for feature in X_train.columns:
        score = train_evaluate(
            X_train.drop(columns=feature),
            X_test.drop(columns=feature)
        )
        print(feature, score)

        results[feature] = baseline - score

    return baseline, results

In [ ]:
ablation(X_train, X_test, y_train, y_test)